# Translon Consensus Analysis: Boundary Agreement Patterns

This notebook analyzes **how different ORF prediction tools agree on feature boundaries**.

Rather than just counting total consensus, we examine:
- **Start-only agreement**: Tools that agree on 5' boundary but disagree on 3' boundary
- **End-only agreement**: Tools that agree on 3' boundary but disagree on 5' boundary  
- **Both boundary agreement**: Tools that agree on both boundaries
- **Tool-specific patterns**: Which tool pairs tend to agree? On which boundaries?

This reveals biological insights about algorithmic differences in boundary detection.

In [ ]:
# Configuration: Set your results directory here
RESULTS_DIR = "/Users/jackt/ensembl-genes-nf/test_consensus_results"

# You can also set a specific sample to focus on (or None for all samples)
FOCUS_SAMPLE = None  # e.g., "sample_1" or None

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print(f"Analyzing results from: {RESULTS_DIR}")

## 1. Load Consensus Data

In [ ]:
def load_consensus_files(results_dir, focus_sample=None):
    """
    Load all consensus TSV files from results directory.
    
    Returns:
        dict: {(sample, query_tool): DataFrame}
    """
    results_dir = Path(results_dir)
    consensus_data = {}
    
    for tsv_file in results_dir.rglob('*.consensus.tsv'):
        parts = tsv_file.stem.split('.')
        if len(parts) >= 3:
            sample_name = parts[0]
            query_tool = parts[1]
            
            # Filter by sample if specified
            if focus_sample and sample_name != focus_sample:
                continue
            
            try:
                df = pd.read_csv(tsv_file, sep='\t')
                consensus_data[(sample_name, query_tool)] = df
                print(f"Loaded {sample_name} / {query_tool}: {len(df)} features")
            except Exception as e:
                print(f"Warning: Failed to load {tsv_file}: {e}")
    
    return consensus_data

consensus_data = load_consensus_files(RESULTS_DIR, FOCUS_SAMPLE)
print(f"\nTotal datasets loaded: {len(consensus_data)}")

## 2. Extract Boundary Agreement Patterns

For each feature in each query tool's results, we examine which other tools agree on:
- **Start only** (`_start_matches > 0` but `_both_matches == 0`)
- **End only** (`_end_matches > 0` but `_both_matches == 0`)  
- **Both** (`_both_matches > 0`)
- **Neither** (all zeros)

In [ ]:
def analyze_boundary_patterns(consensus_data):
    """
    Analyze boundary agreement patterns across all features.
    
    Returns:
        DataFrame with columns: sample, query_tool, other_tool, 
                               start_only, end_only, both, neither, total_features
    """
    pattern_data = []
    
    for (sample, query_tool), df in consensus_data.items():
        # Find match columns
        match_cols = [col for col in df.columns if col.endswith('_both_matches')]
        other_tools = [col.replace('_both_matches', '') for col in match_cols]
        
        for other_tool in other_tools:
            # Extract match counts
            start_matches = df[f'{other_tool}_start_matches']
            end_matches = df[f'{other_tool}_end_matches']
            both_matches = df[f'{other_tool}_both_matches']
            
            # Classify each feature
            start_only = ((start_matches > 0) & (both_matches == 0)).sum()
            end_only = ((end_matches > 0) & (both_matches == 0)).sum()
            both = (both_matches > 0).sum()
            neither = ((start_matches == 0) & (end_matches == 0)).sum()
            
            pattern_data.append({
                'sample': sample,
                'query_tool': query_tool,
                'other_tool': other_tool,
                'start_only': start_only,
                'end_only': end_only,
                'both': both,
                'neither': neither,
                'total_features': len(df)
            })
    
    return pd.DataFrame(pattern_data)

pattern_df = analyze_boundary_patterns(consensus_data)
print(f"\nAnalyzed {len(pattern_df)} tool pairs across samples")
pattern_df.head(10)

## 3. Summary Statistics: Agreement Rates

In [ ]:
# Calculate percentages
pattern_df['pct_start_only'] = 100 * pattern_df['start_only'] / pattern_df['total_features']
pattern_df['pct_end_only'] = 100 * pattern_df['end_only'] / pattern_df['total_features']
pattern_df['pct_both'] = 100 * pattern_df['both'] / pattern_df['total_features']
pattern_df['pct_neither'] = 100 * pattern_df['neither'] / pattern_df['total_features']
pattern_df['pct_any_agreement'] = 100 - pattern_df['pct_neither']

# Summary by tool pair (averaged across samples)
tool_pair_summary = pattern_df.groupby(['query_tool', 'other_tool']).agg({
    'pct_start_only': 'mean',
    'pct_end_only': 'mean',
    'pct_both': 'mean',
    'pct_neither': 'mean',
    'pct_any_agreement': 'mean'
}).round(2)

print("\n=== Average Agreement Rates by Tool Pair ===")
print("\nMost Compatible Tool Pairs (highest 'both' agreement):")
print(tool_pair_summary.sort_values('pct_both', ascending=False).head(10))

## 4. Visualization: Boundary Agreement Patterns

In [ ]:
# Create a stacked bar chart showing agreement patterns
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left plot: Agreement types by tool pair
plot_data = pattern_df.groupby(['query_tool', 'other_tool'])[['start_only', 'end_only', 'both', 'neither']].sum()
plot_data_pct = plot_data.div(plot_data.sum(axis=1), axis=0) * 100

# Get top 15 most interesting pairs (highest non-neither)
plot_data_pct['total_agreement'] = plot_data_pct[['start_only', 'end_only', 'both']].sum(axis=1)
top_pairs = plot_data_pct.nlargest(15, 'total_agreement')

top_pairs[['both', 'start_only', 'end_only', 'neither']].plot(
    kind='barh', 
    stacked=True, 
    ax=axes[0],
    color=['#2ecc71', '#f39c12', '#e74c3c', '#95a5a6'],
    width=0.8
)
axes[0].set_xlabel('Percentage of Features (%)')
axes[0].set_title('Boundary Agreement Patterns\n(Top 15 Tool Pairs)', fontsize=14, fontweight='bold')
axes[0].legend(['Both Boundaries', 'Start Only', 'End Only', 'No Agreement'], loc='lower right')
axes[0].set_xlim(0, 100)

# Right plot: Heatmap of "both" agreement rates
pivot_both = pattern_df.groupby(['query_tool', 'other_tool'])['pct_both'].mean().reset_index()
pivot_matrix = pivot_both.pivot(index='query_tool', columns='other_tool', values='pct_both')

sns.heatmap(pivot_matrix, annot=True, fmt='.1f', cmap='RdYlGn', ax=axes[1], 
            vmin=0, vmax=100, cbar_kws={'label': 'Agreement Rate (%)'})
axes[1].set_title('Full Boundary Agreement Rates\n(Both Start and End)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Other Tool')
axes[1].set_ylabel('Query Tool')

plt.tight_layout()
plt.show()

## 5. Boundary-Specific Agreement Analysis

Let's examine whether tools systematically agree more on **starts** vs **ends**.

In [ ]:
# Calculate total start vs end agreement (including both)
pattern_df['total_start_agreement'] = pattern_df['start_only'] + pattern_df['both']
pattern_df['total_end_agreement'] = pattern_df['end_only'] + pattern_df['both']
pattern_df['pct_total_start'] = 100 * pattern_df['total_start_agreement'] / pattern_df['total_features']
pattern_df['pct_total_end'] = 100 * pattern_df['total_end_agreement'] / pattern_df['total_features']

# Compare start vs end preference
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Start vs End agreement scatter
axes[0].scatter(pattern_df['pct_total_start'], pattern_df['pct_total_end'], 
               alpha=0.6, s=100, c=pattern_df['pct_both'], cmap='viridis')
axes[0].plot([0, 100], [0, 100], 'k--', alpha=0.3, label='Equal agreement')
axes[0].set_xlabel('Start Agreement Rate (%)', fontsize=12)
axes[0].set_ylabel('End Agreement Rate (%)', fontsize=12)
axes[0].set_title('Start vs End Boundary Agreement\n(Color = Both Agreement %)', 
                  fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
cbar = plt.colorbar(axes[0].collections[0], ax=axes[0])
cbar.set_label('Both Agreement (%)', rotation=270, labelpad=20)

# Right: Tool-specific start/end preferences
tool_boundary_pref = pattern_df.groupby('query_tool')[['pct_total_start', 'pct_total_end']].mean()
tool_boundary_pref.plot(kind='bar', ax=axes[1], color=['#3498db', '#e74c3c'], width=0.8)
axes[1].set_ylabel('Agreement Rate (%)', fontsize=12)
axes[1].set_xlabel('Query Tool', fontsize=12)
axes[1].set_title('Start vs End Agreement by Tool\n(Averaged Across Comparisons)', 
                  fontsize=14, fontweight='bold')
axes[1].legend(['Start Agreement', 'End Agreement'])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n=== Start vs End Agreement Summary ===")
print(tool_boundary_pref.round(2))

## 6. Tool Pair Compatibility Matrix

Which tool pairs work best together?

In [ ]:
# Create compatibility score: weighted sum of agreement types
# Both = 100%, Start-only = 50%, End-only = 50%
pattern_df['compatibility_score'] = (
    pattern_df['pct_both'] + 
    0.5 * pattern_df['pct_start_only'] + 
    0.5 * pattern_df['pct_end_only']
)

# Average by tool pair
compatibility = pattern_df.groupby(['query_tool', 'other_tool'])['compatibility_score'].mean().reset_index()
compatibility_matrix = compatibility.pivot(index='query_tool', columns='other_tool', values='compatibility_score')

plt.figure(figsize=(10, 8))
sns.heatmap(compatibility_matrix, annot=True, fmt='.1f', cmap='coolwarm', 
            vmin=0, vmax=100, center=50,
            cbar_kws={'label': 'Compatibility Score'})
plt.title('Tool Pair Compatibility Matrix\n(Higher = More Agreement)', 
         fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Other Tool', fontsize=12)
plt.ylabel('Query Tool', fontsize=12)
plt.tight_layout()
plt.show()

print("\n=== Most Compatible Tool Pairs ===")
top_compat = compatibility.nlargest(10, 'compatibility_score')
for _, row in top_compat.iterrows():
    print(f"{row['query_tool']:15s} <-> {row['other_tool']:15s}  Score: {row['compatibility_score']:.1f}%")

## 7. Context Analysis: What Features Have Partial Agreement?

Let's look at specific examples where tools agree on one boundary but not the other.

In [ ]:
def find_boundary_disagreement_examples(consensus_data, n_examples=5):
    """
    Find examples of features with start-only or end-only agreement.
    """
    examples = {'start_only': [], 'end_only': []}
    
    for (sample, query_tool), df in consensus_data.items():
        if len(examples['start_only']) >= n_examples and len(examples['end_only']) >= n_examples:
            break
            
        match_cols = [col for col in df.columns if col.endswith('_both_matches')]
        other_tools = [col.replace('_both_matches', '') for col in match_cols]
        
        for other_tool in other_tools:
            # Start-only examples
            if len(examples['start_only']) < n_examples:
                start_only_mask = (
                    (df[f'{other_tool}_start_matches'] > 0) & 
                    (df[f'{other_tool}_both_matches'] == 0)
                )
                if start_only_mask.any():
                    example = df[start_only_mask].iloc[0]
                    examples['start_only'].append({
                        'sample': sample,
                        'query_tool': query_tool,
                        'other_tool': other_tool,
                        'chr': example['chr'],
                        'start': example['start_pos'],
                        'end': example['end_pos'],
                        'strand': example['strand'],
                        'start_matches': example[f'{other_tool}_start_matches'],
                        'end_matches': example[f'{other_tool}_end_matches']
                    })
            
            # End-only examples
            if len(examples['end_only']) < n_examples:
                end_only_mask = (
                    (df[f'{other_tool}_end_matches'] > 0) & 
                    (df[f'{other_tool}_both_matches'] == 0)
                )
                if end_only_mask.any():
                    example = df[end_only_mask].iloc[0]
                    examples['end_only'].append({
                        'sample': sample,
                        'query_tool': query_tool,
                        'other_tool': other_tool,
                        'chr': example['chr'],
                        'start': example['start_pos'],
                        'end': example['end_pos'],
                        'strand': example['strand'],
                        'start_matches': example[f'{other_tool}_start_matches'],
                        'end_matches': example[f'{other_tool}_end_matches']
                    })
    
    return examples

examples = find_boundary_disagreement_examples(consensus_data, n_examples=5)

print("\n=== Examples: Start-Only Agreement ===")
print("(Tools agree on 5' boundary but disagree on 3' boundary)\n")
for ex in examples['start_only']:
    print(f"Sample: {ex['sample']}")
    print(f"  Query: {ex['query_tool']} vs Other: {ex['other_tool']}")
    print(f"  Location: {ex['chr']}:{ex['start']}-{ex['end']} ({ex['strand']})")
    print(f"  Start matches: {ex['start_matches']}, End matches: {ex['end_matches']}")
    print()

print("\n=== Examples: End-Only Agreement ===")
print("(Tools agree on 3' boundary but disagree on 5' boundary)\n")
for ex in examples['end_only']:
    print(f"Sample: {ex['sample']}")
    print(f"  Query: {ex['query_tool']} vs Other: {ex['other_tool']}")
    print(f"  Location: {ex['chr']}:{ex['start']}-{ex['end']} ({ex['strand']})")
    print(f"  Start matches: {ex['start_matches']}, End matches: {ex['end_matches']}")
    print()

## 8. Summary Report

Key findings from the consensus analysis.

In [ ]:
print("="*80)
print("TRANSLON CONSENSUS ANALYSIS SUMMARY")
print("="*80)

# Overall statistics
print("\n1. DATASET OVERVIEW")
print(f"   Total samples analyzed: {pattern_df['sample'].nunique()}")
print(f"   Total tools compared: {pattern_df['query_tool'].nunique()}")
print(f"   Total tool pairs: {len(pattern_df)}")
print(f"   Total features analyzed: {pattern_df['total_features'].sum():,}")

# Agreement rates
print("\n2. OVERALL AGREEMENT RATES (Average across all tool pairs)")
print(f"   Both boundaries agree:  {pattern_df['pct_both'].mean():.1f}%")
print(f"   Start only agrees:      {pattern_df['pct_start_only'].mean():.1f}%")
print(f"   End only agrees:        {pattern_df['pct_end_only'].mean():.1f}%")
print(f"   No agreement:           {pattern_df['pct_neither'].mean():.1f}%")

# Best/worst tool pairs
best_pair = pattern_df.nlargest(1, 'pct_both').iloc[0]
worst_pair = pattern_df.nsmallest(1, 'pct_both').iloc[0]

print("\n3. TOOL PAIR PERFORMANCE")
print(f"   Best agreement:  {best_pair['query_tool']} <-> {best_pair['other_tool']} ({best_pair['pct_both']:.1f}%)")
print(f"   Worst agreement: {worst_pair['query_tool']} <-> {worst_pair['other_tool']} ({worst_pair['pct_both']:.1f}%)")

# Boundary preferences
print("\n4. BOUNDARY-SPECIFIC AGREEMENT")
print(f"   Average start agreement: {pattern_df['pct_total_start'].mean():.1f}%")
print(f"   Average end agreement:   {pattern_df['pct_total_end'].mean():.1f}%")
if pattern_df['pct_total_start'].mean() > pattern_df['pct_total_end'].mean():
    print("   → Tools agree more on START boundaries")
else:
    print("   → Tools agree more on END boundaries")

print("\n" + "="*80)

## 9. Export Summary Table

Save the full pattern analysis to TSV for further investigation.

In [ ]:
# Export full pattern data
output_file = Path(RESULTS_DIR) / "consensus_boundary_analysis.tsv"
pattern_df.to_csv(output_file, sep='\t', index=False)
print(f"\nFull analysis exported to: {output_file}")

# Export tool pair compatibility matrix
compat_file = Path(RESULTS_DIR) / "tool_compatibility_matrix.tsv"
compatibility_matrix.to_csv(compat_file, sep='\t')
print(f"Compatibility matrix exported to: {compat_file}")